In [ ]:
"""
sandbox_time.ipynb

A sandbox to develop a time-resolved class.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

## init

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder

encoder = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
)

encoder_mb = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
    strategy_filter="mb",
)

encoder_mf = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
    strategy_filter="mf",
)

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(3, 5), tight_layout=True)
encoder.plot_r2_distro(axes[0])
encoder_mb.plot_r2_distro(axes[1])
encoder_mf.plot_r2_distro(axes[2])

axes[0].set_xlim([-0.05, 0.8])
axes[1].set_xlim([-0.05, 0.8])
axes[2].set_xlim([-0.05, 0.8])

axes[0].set_xlabel(r"$r^2$, full")
axes[1].set_xlabel(r"$r^2$, mb")
axes[2].set_xlabel(r"$r^2$, mf")

In [ ]:
encoder.verify()
encoder_mb.verify()
encoder_mf.verify()

In [ ]:
encoder.view_fits(model="baseline")

In [ ]:
encoder.view_fits()

In [ ]:
encoder.view_weights()

## trajectories

In [ ]:
from sg.models import Bootstrapper

bs_mb = Bootstrapper(
    subj_id,
    sess_id,
    make_tre(StrategyEncoder, tr_type="dme"),
    n=20,
    norm=True,
    stepsize_s=0.25,
    strategy_filter="mb",
)
bs_mf = Bootstrapper(
    subj_id,
    sess_id,
    make_tre(StrategyEncoder, tr_type="dme"),
    n=20,
    norm=True,
    stepsize_s=0.25,
    strategy_filter="mf",
)

bs_mb.get_bweight_stats()
bs_mf.get_bweight_stats()

In [ ]:
from squiggs.renderers import StrategyWeightPETHRenderer
from squiggs.neuron_viewer import NeuronViewer
from core.data import get_psths_cond, tv_pos_neg

mode = "response"
regressor = mode
reg = "DMS"
# rewd 33, dms
# resp 30, 36

r = StrategyWeightPETHRenderer(
    bootstrapper_mb=bs_mb,
    bootstrapper_mf=bs_mf,
    encoder_mb=encoder_mb,
    encoder_mf=encoder_mf,
    reg=reg,
    regressor=regressor,
    values=(tv_pos_neg[regressor]["pos"], tv_pos_neg[regressor]["neg"]),
    peths_mb=get_psths_cond(encoder_mb.psths[reg], encoder_mb.trial_data, mode=mode),
    peths_mf=get_psths_cond(encoder_mf.psths[reg], encoder_mf.trial_data, mode=mode),
    pres=encoder.tpre,
    posts=encoder.tpost,
    binwidth_s=encoder.stepsize_s,
    tbin_centers=encoder.tbin_centers,
)


nv = NeuronViewer(num_units=encoder.psths[reg].shape[0], render_func=r)

## response only

In [ ]:
from sg.models import make_tre, Encoder

encoder_response = make_tre(Encoder)(
    subj_id,
    sess_id,
    tv_keys=["response"],
    norm=False,
    stepsize_s=0.1,
)

In [ ]:
encoder_response.view_weights()